# AI 체력 코치 — Qwen3 배포본 RAG 검증·갱신 노트북

배포본 SQLite와 실행용 Chroma를 함께 검증합니다. Step 5는 별도 후보 폴더에서 전체 문서 ID·내용·메타데이터를 비교해 변경분을 갱신합니다.


## 실행 구조

배포 폴더 `artifacts/rag_documents.sqlite` → 번들 SentenceTransformer → `C:/ai_fitness_qwen3_runtime/chroma` → Qwen3 하네스 → 답변·출처.

Step 5는 `chroma_candidate`에만 갱신합니다. 검증 후 서버를 중지하고 배포 인덱스와 실행 캐시를 함께 교체해야 합니다. 노트북은 현재 서비스 인덱스를 자동 교체하지 않습니다. 기존 해시 벡터와 SentenceTransformer 벡터는 혼용하지 않습니다.


## Step 0. 현재 경로

원문 DB: `./artifacts/rag_documents.sqlite`

실행 인덱스: `C:/ai_fitness_qwen3_runtime/chroma`

재구축 후보: `C:/ai_fitness_qwen3_runtime/chroma_candidate`


In [ ]:
# 필수 패키지는 Step 4에서 현재 커널에 자동 설치합니다.
# 이 셀은 별도로 실행할 필요가 없습니다.

## Step 1. Qwen3 배포 DB 연결

현재 배포 폴더의 SQLite를 사용합니다. C:/ai_fitness_local의 과거 DB를 자동 선택하지 않습니다. 원본 전처리를 다시 할 때만 `RUN_PREPROCESSING=True`로 바꾸고 입력 경로를 확인하세요.


In [ ]:
import os, sys, json, gzip, csv, sqlite3
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "tools/fitness_web_server.py").is_file():
    raise FileNotFoundError("Qwen3 배포 폴더에서 노트북을 실행하세요.")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
RAG_DATABASE_PATH = ARTIFACTS_DIR / "rag_documents.sqlite"
RUNTIME_ROOT = Path(os.environ.get("AI_FITNESS_RUNTIME_ROOT", "C:/ai_fitness_qwen3_runtime"))
ARTIFACTS_READY = RAG_DATABASE_PATH.is_file()
if not ARTIFACTS_READY:
    raise FileNotFoundError(RAG_DATABASE_PATH)
IN_COLAB = False
RUN_PREPROCESSING = False
RUN_VECTOR_BUILD = False
SOURCE_ROOT = PROJECT_ROOT / "source_data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
EXPECTED_FOLDERS = []
print("Qwen3 원문 DB:", RAG_DATABASE_PATH)
print("실행 인덱스:", RUNTIME_ROOT / "chroma")


### 1-1. DB 산출물·원본 데이터 확인

기본으로 DB 산출물을 확인합니다. `RUN_PREPROCESSING=True`인 경우에만 6개 원본 폴더도 검사합니다.

In [ ]:
artifact_paths = {
    "rag_documents.sqlite": RAG_DATABASE_PATH,
    "analytics.duckdb": ARTIFACTS_DIR / "analytics.duckdb",
    "manifest.json": ARTIFACTS_DIR / "manifest.json",
    "quality_report.json": ARTIFACTS_DIR / "quality_report.json",
}
artifact_rows = []
for name, path in artifact_paths.items():
    required = name == "rag_documents.sqlite"
    exists = path is not None and path.is_file()
    artifact_rows.append({
        "파일": name,
        "필수": required,
        "존재": exists,
        "크기(MB)": round(path.stat().st_size / 1024**2, 2) if exists else None,
        "경로": str(path) if path is not None else "",
    })
display(pd.DataFrame(artifact_rows))


def inventory_source(root: Path) -> pd.DataFrame:
    rows = []
    for folder_name in EXPECTED_FOLDERS:
        folder = root / folder_name
        files = [path for path in folder.rglob("*") if path.is_file()] if folder.exists() else []
        rows.append({
            "데이터 폴더": folder_name,
            "존재": folder.exists(),
            "파일 수": len(files),
            "용량(MB)": round(sum(path.stat().st_size for path in files) / 1024**2, 2),
            "확장자": ", ".join(sorted({path.suffix.lower() for path in files})),
        })
    return pd.DataFrame(rows)


inventory_df = inventory_source(SOURCE_ROOT)
missing_folders = inventory_df.loc[~inventory_df["존재"], "데이터 폴더"].tolist()
if RUN_PREPROCESSING:
    display(inventory_df)
    if missing_folders:
        print("원본 경로를 확인하세요. 누락 폴더:", missing_folders)
    else:
        print("6개 원본 데이터 폴더 확인 완료")
else:
    print("기존 DB 사용 모드: 6개 원본 폴더는 필요하지 않습니다.")

## Step 2. 데이터 전처리 정책

### 격리하는 행

- 나이 누락 또는 숫자 변환 실패
- 나이가 3세 미만 또는 120세 초과
- 성별 코드가 `M`, `F` 이외
- 측정일·측정월 형식 오류 또는 미래 날짜
- 처방 내용 누락
- 측정값이 숫자가 아니거나 유한하지 않은 경우
- 체력측정 행에 유효한 숫자 측정값이 하나도 없는 경우
- 측정값 절댓값이 1,000,000을 초과하는 명백한 오류

### 경고만 표시하고 유지하는 행

- 센터명·주소·연령군 같은 보조 필드 누락
- 연령과 연령대 코드 불일치
- 일부 측정항목의 비수치 값

### 중요한 제한

`item_f001` 같은 측정항목 코드의 정확한 의미와 단위가 확정되지 않은 상태에서는 통계적 극단값만으로 행을 삭제하지 않습니다. 항목 정의서를 확보하면 `항목별 허용 범위`를 추가해 재처리해야 합니다. 현재는 재현 가능하고 보수적인 하드 오류 규칙만 적용합니다.


### 2-1. 전처리 구현 코드

아래 셀에는 대용량 JSON 스트리밍, CSV 정규화, 결측·범위 검사, 정상/격리 분리, 품질 통계, 검색 문서 생성 코드가 모두 들어 있습니다. 셀은 길어서 기본적으로 접어두었으며, 팀원이 필요할 때 펼쳐 검토할 수 있습니다.


In [ ]:
from __future__ import annotations

import argparse
import csv
import gzip
import hashlib
import html
import json
import math
import re
import sqlite3
from collections import Counter
from dataclasses import dataclass, field
from datetime import date, datetime
from pathlib import Path
from typing import Any, Iterable, Iterator, TextIO

TAG_RE = re.compile(r"<[^>]+>")
SPACE_RE = re.compile(r"\s+")
ITEM_RE = re.compile(r"^item_f\d+$")
MIN_AGE, MAX_AGE, MIN_YEAR = 3, 120, 2000
ALLOWED_SEX = {"M", "F"}


def clean_text(value: Any) -> str:
    if value is None:
        return ""
    value = html.unescape(str(value)).replace("\u0000", " ")
    return SPACE_RE.sub(" ", TAG_RE.sub(" ", value)).strip()


def parse_int(value: Any) -> int | None:
    try:
        number = float(clean_text(value))
    except ValueError:
        return None
    return int(number) if math.isfinite(number) and number.is_integer() else None


def parse_float(value: Any) -> float | None:
    try:
        number = float(clean_text(value))
    except ValueError:
        return None
    return number if math.isfinite(number) else None


def parse_date(value: Any, fmt: str) -> str | None:
    try:
        parsed = datetime.strptime(clean_text(value), fmt)
    except ValueError:
        return None
    if parsed.year < MIN_YEAR or parsed.date() > date.today():
        return None
    return parsed.strftime("%Y-%m-%d" if fmt == "%Y%m%d" else "%Y-%m")


def iter_json_array(path: Path, chunk_size: int = 4 * 1024 * 1024) -> Iterator[dict[str, Any]]:
    """2GB급 최상위 JSON 배열을 메모리에 올리지 않고 읽는다."""
    decoder, buffer, started, eof = json.JSONDecoder(), "", False, False
    with path.open("r", encoding="utf-8-sig") as stream:
        while True:
            if not eof:
                chunk = stream.read(chunk_size)
                if chunk:
                    buffer += chunk
                else:
                    eof = True
            pos = 0
            if not started:
                while pos < len(buffer) and buffer[pos].isspace():
                    pos += 1
                if pos >= len(buffer):
                    if eof:
                        return
                    buffer = ""
                    continue
                if buffer[pos] != "[":
                    raise ValueError(f"최상위 JSON 배열이 아닙니다: {path}")
                started, pos = True, pos + 1
            decoded = False
            while True:
                while pos < len(buffer) and (buffer[pos].isspace() or buffer[pos] == ","):
                    pos += 1
                if pos < len(buffer) and buffer[pos] == "]":
                    return
                if pos >= len(buffer):
                    break
                try:
                    item, end = decoder.raw_decode(buffer, pos)
                except json.JSONDecodeError:
                    break
                if isinstance(item, dict):
                    yield item
                pos, decoded = end, True
            buffer = buffer[pos:]
            if eof:
                if buffer.strip() not in {"", "]"}:
                    raise ValueError(f"불완전한 JSON 배열입니다: {path}")
                return
            if not decoded and len(buffer) > chunk_size * 8:
                raise ValueError(f"비정상적으로 큰 JSON 객체가 있습니다: {path}")


@dataclass
class QualityStats:
    dataset: str
    source_file: str
    input_rows: int = 0
    clean_rows: int = 0
    quarantine_rows: int = 0
    warning_rows: int = 0
    reason_counts: Counter[str] = field(default_factory=Counter)
    warning_counts: Counter[str] = field(default_factory=Counter)

    def as_dict(self) -> dict[str, Any]:
        return {
            "dataset": self.dataset, "source_file": self.source_file,
            "input_rows": self.input_rows, "clean_rows": self.clean_rows,
            "quarantine_rows": self.quarantine_rows, "warning_rows": self.warning_rows,
            "reason_counts": dict(sorted(self.reason_counts.items())),
            "warning_counts": dict(sorted(self.warning_counts.items())),
        }


class RagStore:
    def __init__(self, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        self.connection = sqlite3.connect(path)
        self.connection.execute("PRAGMA journal_mode=WAL")
        self.connection.execute("PRAGMA synchronous=NORMAL")
        self.connection.execute(
            """CREATE TABLE IF NOT EXISTS documents (
            id TEXT PRIMARY KEY, doc_key TEXT UNIQUE NOT NULL, dataset TEXT NOT NULL,
            title TEXT NOT NULL, content TEXT NOT NULL, metadata_json TEXT NOT NULL,
            occurrence_count INTEGER NOT NULL DEFAULT 1, vector_row INTEGER)"""
        )
        self.connection.execute("CREATE INDEX IF NOT EXISTS idx_documents_dataset ON documents(dataset)")

    def add(self, dataset: str, title: str, content: str, metadata: dict[str, Any], key_parts: Iterable[Any]) -> None:
        doc_key = "\u241f".join([dataset, *[clean_text(part).lower() for part in key_parts]])
        doc_id = hashlib.sha256(doc_key.encode("utf-8")).hexdigest()[:24]
        self.connection.execute(
            """INSERT INTO documents(id, doc_key, dataset, title, content, metadata_json, occurrence_count)
            VALUES (?, ?, ?, ?, ?, ?, 1)
            ON CONFLICT(doc_key) DO UPDATE SET occurrence_count=occurrence_count+1""",
            (doc_id, doc_key, dataset, title, content, json.dumps(metadata, ensure_ascii=False, sort_keys=True)),
        )

    def commit(self) -> None:
        self.connection.commit()

    def close(self) -> None:
        self.connection.commit()
        self.connection.close()


class JsonlWriter:
    def __init__(self, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        self.stream: TextIO = gzip.open(path, "wt", encoding="utf-8", newline="")

    def write(self, item: dict[str, Any]) -> None:
        self.stream.write(json.dumps(item, ensure_ascii=False, sort_keys=True) + "\n")

    def close(self) -> None:
        self.stream.close()


def open_csv_writer(path: Path, fields: list[str]) -> tuple[TextIO, csv.DictWriter]:
    path.parent.mkdir(parents=True, exist_ok=True)
    stream: TextIO = gzip.open(path, "wt", encoding="utf-8-sig", newline="") if path.suffix == ".gz" else path.open("w", encoding="utf-8-sig", newline="")
    writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="ignore")
    writer.writeheader()
    return stream, writer


def normalize_row(row: dict[str, Any]) -> dict[str, str]:
    return {str(key).strip(): clean_text(value) for key, value in row.items() if key is not None}


def prescription_doc(row: dict[str, str], disability: bool):
    prescription = row.get("MVM_PRSCRPTN_CN", "")
    dataset = "disability_prescription" if disability else "general_prescription"
    title = "장애인 맞춤 운동처방" if disability else "국민체력100 운동처방"
    labels = [f"대상 연령군: {row.get('AGE_FLAG_NM') or '정보 없음'}", f"성별: {row.get('SEXDSTN_FLAG_CD') or '정보 없음'}", f"측정 장소: {row.get('MESURE_PLACE_FLAG_NM') or '정보 없음'}"]
    if disability:
        labels += [f"장애유형: {row.get('TROBL_TY_NM') or '정보 없음'}", f"장애 세부유형: {row.get('TROBL_DETAIL_NM') or '정보 없음'}", f"장애등급: {row.get('TROBL_GRAD_NM') or '정보 없음'}"]
    else:
        labels.append(f"체력 인증등급: {row.get('COAW_FLAG_NM') or '정보 없음'}")
    content = f"{title}\n" + "\n".join(labels) + f"\n처방 내용: {prescription}"
    metadata = {"source": "국민체력100", "dataset": dataset, "age_group": row.get("AGE_FLAG_NM", ""), "sex": row.get("SEXDSTN_FLAG_CD", ""), "place": row.get("MESURE_PLACE_FLAG_NM", ""), "source_file": row.get("source_file", "")}
    if disability:
        metadata.update({"disability_type": row.get("TROBL_TY_NM", ""), "disability_detail": row.get("TROBL_DETAIL_NM", ""), "disability_grade": row.get("TROBL_GRAD_NM", "")})
        key = [prescription, metadata["age_group"], metadata["sex"], metadata["disability_type"], metadata["disability_detail"], metadata["disability_grade"]]
    else:
        metadata["certification_grade"] = row.get("COAW_FLAG_NM", "")
        key = [prescription, metadata["age_group"], metadata["sex"], metadata["certification_grade"]]
    return title, content, metadata, key


def process_prescriptions(source: Path, output: Path, store: RagStore, disability: bool) -> list[QualityStats]:
    dataset = "disability_prescription" if disability else "general_prescription"
    files = sorted(source.glob("*.csv"))
    if not files:
        raise FileNotFoundError(source)
    with files[0].open("r", encoding="utf-8-sig", newline="") as sample:
        base_fields = list(csv.DictReader(sample).fieldnames or [])
    extras = ["source_dataset", "source_file", "source_row", "measurement_date_iso", "validation_status", "warning_codes", "invalid_reason"]
    clean_stream, clean_writer = open_csv_writer(output / f"{dataset}_clean.csv.gz", base_fields + extras)
    bad_stream, bad_writer = open_csv_writer(output / f"{dataset}_quarantine.csv.gz", base_fields + extras)
    reports: list[QualityStats] = []
    try:
        for path in files:
            stats = QualityStats(dataset, path.name)
            with path.open("r", encoding="utf-8-sig", newline="") as stream:
                reader = csv.DictReader(stream)
                if list(reader.fieldnames or []) != base_fields:
                    raise ValueError(f"월별 스키마 불일치: {path.name}")
                for row_no, raw in enumerate(reader, 2):
                    stats.input_rows += 1
                    row, errors, warnings = normalize_row(raw), [], []
                    age = parse_int(row.get("MESURE_AGE_CO"))
                    if age is None:
                        errors.append("INVALID_OR_MISSING_AGE")
                    elif not MIN_AGE <= age <= MAX_AGE:
                        errors.append("AGE_OUT_OF_RANGE")
                    else:
                        row["MESURE_AGE_CO"] = str(age)
                    sex = row.get("SEXDSTN_FLAG_CD", "").upper()
                    row["SEXDSTN_FLAG_CD"] = sex
                    if sex not in ALLOWED_SEX:
                        errors.append("INVALID_OR_MISSING_SEX")
                    iso_date = parse_date(row.get("MESURE_DE"), "%Y%m%d")
                    if not iso_date:
                        errors.append("INVALID_OR_MISSING_DATE")
                    if not row.get("MVM_PRSCRPTN_CN"):
                        errors.append("MISSING_PRESCRIPTION")
                    if not row.get("CNTER_NM"):
                        warnings.append("MISSING_CENTER")
                    if not row.get("AGE_FLAG_NM"):
                        warnings.append("MISSING_AGE_GROUP")
                    if disability and not row.get("TROBL_TY_NM"):
                        warnings.append("MISSING_DISABILITY_TYPE")
                    row.update({"source_dataset": dataset, "source_file": path.name, "source_row": row_no, "measurement_date_iso": iso_date or "", "validation_status": "quarantine" if errors else ("warning" if warnings else "clean"), "warning_codes": "|".join(sorted(set(warnings))), "invalid_reason": "|".join(sorted(set(errors)))})
                    if errors:
                        bad_writer.writerow(row); stats.quarantine_rows += 1; stats.reason_counts.update(errors)
                    else:
                        clean_writer.writerow(row); stats.clean_rows += 1
                        if warnings:
                            stats.warning_rows += 1; stats.warning_counts.update(warnings)
                        store.add(dataset, *prescription_doc(row, disability))
                    if stats.input_rows % 10000 == 0:
                        store.commit()
            reports.append(stats); store.commit()
    finally:
        clean_stream.close(); bad_stream.close()
    return reports


def process_center_stats(source: Path, output: Path) -> list[QualityStats]:
    path = next(iter(sorted(source.glob("*.csv"))), None)
    if path is None:
        raise FileNotFoundError(source)
    with path.open("r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream); fields = list(reader.fieldnames or [])
        extras = ["source_dataset", "source_file", "source_row", "validation_status", "warning_codes", "invalid_reason"]
        clean_s, clean_w = open_csv_writer(output / "center_statistics_clean.csv", fields + extras)
        bad_s, bad_w = open_csv_writer(output / "center_statistics_quarantine.csv", fields + extras)
        stats = QualityStats("center_statistics", path.name)
        try:
            count_fields = [name for name in fields if name.endswith("_CO")]
            for row_no, raw in enumerate(reader, 2):
                stats.input_rows += 1
                row, errors, warnings = normalize_row(raw), [], []
                if not row.get("CNTER_NM"):
                    errors.append("MISSING_CENTER")
                year = parse_int(row.get("MESURE_YEAR"))
                if year is None or not MIN_YEAR <= year <= date.today().year:
                    errors.append("INVALID_YEAR")
                for field_name in count_fields:
                    number = parse_int(row.get(field_name))
                    if number is None or number < 0:
                        errors.append(f"INVALID_COUNT:{field_name}")
                    else:
                        row[field_name] = str(number)
                total, male, female = parse_int(row.get("ALL_MESURE_CAS_CO")), parse_int(row.get("MALE_MBER_CO")), parse_int(row.get("FEMALE_MBER_CO"))
                if None not in (total, male, female) and male + female > total:
                    warnings.append("SEX_COUNT_EXCEEDS_TOTAL_MEASUREMENTS")
                row.update({"source_dataset": "center_statistics", "source_file": path.name, "source_row": row_no, "validation_status": "quarantine" if errors else ("warning" if warnings else "clean"), "warning_codes": "|".join(warnings), "invalid_reason": "|".join(errors)})
                if errors:
                    bad_w.writerow(row); stats.quarantine_rows += 1; stats.reason_counts.update(errors)
                else:
                    clean_w.writerow(row); stats.clean_rows += 1
                    if warnings:
                        stats.warning_rows += 1; stats.warning_counts.update(warnings)
        finally:
            clean_s.close(); bad_s.close()
    return [stats]


def load_items(path: Path) -> list[dict[str, Any]]:
    with path.open("r", encoding="utf-8-sig") as stream:
        payload = json.load(stream)
    items = payload.get("items", []) if isinstance(payload, dict) else payload
    if isinstance(items, dict):
        items = items.get("item", items)
    if isinstance(items, dict):
        items = [items]
    return [item for item in items if isinstance(item, dict)]


def video_doc(row: dict[str, Any], source_file: str):
    name = clean_text(row.get("trng_nm") or row.get("vdo_ttl") or row.get("title"))
    desc = clean_text(row.get("vdo_desc") or row.get("description"))
    title = name or desc[:80] or "국민체력100 운동 동영상"
    fields = [("운동명", name), ("설명", desc), ("대상 연령군", clean_text(row.get("aggrp_nm"))), ("난이도", clean_text(row.get("ftns_lvl_nm"))), ("운동 장소", clean_text(row.get("trng_plc_nm"))), ("운동 부위", clean_text(row.get("trng_mscl_part"))), ("주요 근육", clean_text(row.get("trng_mscl_nm") or row.get("trng_mscl_zn_nm"))), ("운동 도구", clean_text(row.get("tool_nm"))), ("반복 횟수", clean_text(row.get("rptt_tcnt_nm"))), ("권장 주기", clean_text(row.get("ecrg_cycl_nm"))), ("영상 URL", clean_text(row.get("file_url")))]
    content = "국민체력100 공식 운동 콘텐츠\n" + "\n".join(f"{label}: {value}" for label, value in fields if value)
    metadata = {"source": "국민체력100 동영상 정보", "dataset": "video_content", "source_file": source_file, "row_num": clean_text(row.get("row_num")), "age_group": clean_text(row.get("aggrp_nm")), "difficulty": clean_text(row.get("ftns_lvl_nm")), "place": clean_text(row.get("trng_plc_nm")), "video_url": clean_text(row.get("file_url"))}
    return title, content, metadata, [source_file, metadata["row_num"], title, desc]


def process_videos(source: Path, output: Path, store: RagStore) -> list[QualityStats]:
    clean, bad, reports = JsonlWriter(output / "video_content_clean.jsonl.gz"), JsonlWriter(output / "video_content_quarantine.jsonl.gz"), []
    try:
        for path in sorted(source.glob("*.json")):
            stats = QualityStats("video_content", path.name)
            for row_no, raw in enumerate(load_items(path), 1):
                stats.input_rows += 1
                row = {key: clean_text(value) if isinstance(value, str) else value for key, value in raw.items()}
                missing = not clean_text(row.get("trng_nm") or row.get("vdo_ttl") or row.get("title")) and not clean_text(row.get("vdo_desc") or row.get("description"))
                row.update({"source_dataset": "video_content", "source_file": path.name, "source_row": row_no})
                if missing:
                    row["invalid_reason"] = ["MISSING_TITLE_AND_DESCRIPTION"]; bad.write(row); stats.quarantine_rows += 1; stats.reason_counts.update(["MISSING_TITLE_AND_DESCRIPTION"])
                else:
                    clean.write(row); stats.clean_rows += 1; store.add("video_content", *video_doc(row, path.name))
            reports.append(stats); store.commit()
    finally:
        clean.close(); bad.close()
    return reports


def process_center_counts(source: Path, output: Path) -> list[QualityStats]:
    clean, bad, reports = JsonlWriter(output / "center_measurements_clean.jsonl.gz"), JsonlWriter(output / "center_measurements_quarantine.jsonl.gz"), []
    try:
        for path in sorted(source.glob("*.json")):
            stats = QualityStats("center_measurements", path.name)
            for row_no, raw in enumerate(load_items(path), 1):
                stats.input_rows += 1
                row, errors, warnings = dict(raw), [], []
                if not clean_text(row.get("center_nm")):
                    errors.append("MISSING_CENTER")
                month = parse_date(row.get("test_ym"), "%Y%m")
                if not month:
                    errors.append("INVALID_OR_MISSING_MONTH")
                count = parse_int(row.get("test_cnt"))
                if count is None or count < 0:
                    errors.append("INVALID_MEASUREMENT_COUNT")
                if not clean_text(row.get("center_addr1")):
                    warnings.append("MISSING_PRIMARY_ADDRESS")
                row.update({"source_dataset": "center_measurements", "source_file": path.name, "source_row": row_no, "test_month_iso": month or "", "validation_status": "quarantine" if errors else ("warning" if warnings else "clean"), "warning_codes": warnings})
                if errors:
                    row["invalid_reason"] = errors; bad.write(row); stats.quarantine_rows += 1; stats.reason_counts.update(errors)
                else:
                    clean.write(row); stats.clean_rows += 1
                    if warnings:
                        stats.warning_rows += 1; stats.warning_counts.update(warnings)
            reports.append(stats)
    finally:
        clean.close(); bad.close()
    return reports


def measurement_doc(row: dict[str, Any], source_file: str):
    prescription = clean_text(row.get("pres_note"))
    if not prescription:
        return None
    age_group, sex, grade = clean_text(row.get("age_gbn")), clean_text(row.get("test_sex")).upper(), clean_text(row.get("cert_gbn"))
    title = "체력측정 결과 기반 운동처방"
    content = f"{title}\n대상 연령군: {age_group or '정보 없음'}\n성별: {sex or '정보 없음'}\n체력 인증등급: {grade or '정보 없음'}\n처방 내용: {prescription}"
    metadata = {"source": "국민체력100 체력측정결과", "dataset": "measurement_prescription", "source_file": source_file, "age_group": age_group, "sex": sex, "certification_grade": grade}
    return title, content, metadata, [prescription, age_group, sex, grade]


def process_measurements(source: Path, output: Path, store: RagStore):
    clean, bad, reports = JsonlWriter(output / "fitness_measurements_clean.jsonl.gz"), JsonlWriter(output / "fitness_measurements_quarantine.jsonl.gz"), []
    summaries: dict[str, dict[str, float | int | None]] = {}
    try:
        for path in sorted(source.glob("*.json")):
            stats = QualityStats("fitness_measurements", path.name)
            for row_no, raw in enumerate(iter_json_array(path), 1):
                stats.input_rows += 1
                row, errors, warnings = dict(raw), [], []
                age = parse_int(row.get("age_degree"))
                if age is None:
                    errors.append("INVALID_OR_MISSING_AGE")
                elif not MIN_AGE <= age <= MAX_AGE:
                    errors.append("AGE_OUT_OF_RANGE")
                sex = clean_text(row.get("test_sex")).upper()
                if sex not in ALLOWED_SEX:
                    errors.append("INVALID_OR_MISSING_SEX")
                month = parse_date(row.get("test_ym"), "%Y%m")
                if not month:
                    errors.append("INVALID_OR_MISSING_MONTH")
                numeric_count = 0
                for key in sorted(k for k in row if ITEM_RE.match(k)):
                    if not clean_text(row.get(key)):
                        row[key] = None; continue
                    value = parse_float(row.get(key))
                    if value is None:
                        row[key] = None; warnings.append(f"NON_NUMERIC_MEASUREMENT:{key}"); continue
                    if abs(value) > 1_000_000:
                        errors.append(f"ABSURD_MEASUREMENT:{key}"); continue
                    row[key], numeric_count = value, numeric_count + 1
                    summary = summaries.setdefault(key, {"count": 0, "min": None, "max": None, "mean": 0.0, "m2": 0.0})
                    summary["count"] = int(summary["count"]) + 1
                    count, mean = int(summary["count"]), float(summary["mean"])
                    delta = value - mean
                    summary["mean"] = mean + delta / count
                    summary["m2"] = float(summary["m2"]) + delta * (value - float(summary["mean"]))
                    summary["min"] = value if summary["min"] is None else min(float(summary["min"]), value)
                    summary["max"] = value if summary["max"] is None else max(float(summary["max"]), value)
                if not numeric_count:
                    errors.append("NO_NUMERIC_MEASUREMENTS")
                age_class = parse_int(row.get("age_class"))
                if age is not None and age_class is not None and not age_class <= age <= age_class + 9:
                    warnings.append("AGE_CLASS_MISMATCH")
                row.update({"source_dataset": "fitness_measurements", "source_file": path.name, "source_row": row_no, "age": age, "test_sex": sex, "test_month_iso": month or "", "validation_status": "quarantine" if errors else ("warning" if warnings else "clean"), "warning_codes": sorted(set(warnings))})
                if errors:
                    row["invalid_reason"] = sorted(set(errors)); bad.write(row); stats.quarantine_rows += 1; stats.reason_counts.update(errors)
                else:
                    clean.write(row); stats.clean_rows += 1
                    if warnings:
                        stats.warning_rows += 1; stats.warning_counts.update(warnings)
                    doc = measurement_doc(row, path.name)
                    if doc:
                        store.add("measurement_prescription", *doc)
                if stats.input_rows % 25000 == 0:
                    store.commit(); print(f"[fitness_measurements] {stats.input_rows:,}행 처리", flush=True)
            reports.append(stats); store.commit()
    finally:
        clean.close(); bad.close()
    for summary in summaries.values():
        count = int(summary["count"])
        summary["std"] = math.sqrt(float(summary["m2"]) / (count - 1)) if count > 1 else 0.0
        del summary["m2"]
    return reports, summaries


def export_rag(database: Path, output: Path) -> int:
    connection, count = sqlite3.connect(database), 0
    with gzip.open(output, "wt", encoding="utf-8") as stream:
        for values in connection.execute("SELECT id,dataset,title,content,metadata_json,occurrence_count FROM documents ORDER BY dataset,id"):
            doc_id, dataset, title, content, metadata_json, occurrences = values
            stream.write(json.dumps({"id": doc_id, "dataset": dataset, "title": title, "text": content, "metadata": json.loads(metadata_json), "occurrence_count": occurrences}, ensure_ascii=False, sort_keys=True) + "\n")
            count += 1
    connection.close()
    return count


def write_reports(output: Path, reports: list[QualityStats], measurements: dict[str, Any], rag_count: int) -> None:
    report_dir = output / "reports"; report_dir.mkdir(parents=True, exist_ok=True)
    rows = [report.as_dict() for report in reports]
    input_count, clean_count, bad_count = sum(r["input_rows"] for r in rows), sum(r["clean_rows"] for r in rows), sum(r["quarantine_rows"] for r in rows)
    payload = {"generated_at": datetime.now().isoformat(timespec="seconds"), "rules": {"age_range": [MIN_AGE, MAX_AGE], "allowed_sex": sorted(ALLOWED_SEX), "measurement_policy": "항목 코드/단위 미확정으로 비수치·비유한값과 절대값 1,000,000 초과만 하드 오류. 통계적 극단값은 삭제하지 않음.", "duplicate_policy": "개인 식별자가 없어 원본 중복은 유지하고 RAG 문서에서만 동일 조건·처방을 집계."}, "totals": {"input_rows": input_count, "clean_rows": clean_count, "quarantine_rows": bad_count, "rag_documents": rag_count, "row_conservation_ok": input_count == clean_count + bad_count}, "files": rows, "measurement_items": measurements}
    (report_dir / "quality_report.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    with (report_dir / "data_quality_report.csv").open("w", encoding="utf-8-sig", newline="") as stream:
        fields = ["dataset", "source_file", "input_rows", "clean_rows", "quarantine_rows", "warning_rows", "reason_counts", "warning_counts"]
        writer = csv.DictWriter(stream, fieldnames=fields); writer.writeheader()
        for item in rows:
            row = dict(item); row["reason_counts"] = json.dumps(row["reason_counts"], ensure_ascii=False, sort_keys=True); row["warning_counts"] = json.dumps(row["warning_counts"], ensure_ascii=False, sort_keys=True); writer.writerow(row)


def run(source: Path, output: Path) -> None:
    source, output = source.resolve(), output.resolve()
    processed, rag_dir = output / "processed", output / "rag"
    processed.mkdir(parents=True, exist_ok=True); rag_dir.mkdir(parents=True, exist_ok=True)
    database = rag_dir / "rag_documents.sqlite"
    if database.exists():
        database.unlink()
    store, reports, summary = RagStore(database), [], {}
    try:
        reports += process_prescriptions(source / "체력 측정별 운동처방 데이터", processed, store, False)
        reports += process_prescriptions(source / "장애인 체력 측정별 운동처방 데이터", processed, store, True)
        reports += process_center_stats(source / "센터별 통계정보", processed)
        reports += process_videos(source / "국민체력100 동영상 정보", processed, store)
        reports += process_center_counts(source / "체력인증센터 측정건수", processed)
        measurement_reports, summary = process_measurements(source / "국민체력측정결과", processed, store)
        reports += measurement_reports
    finally:
        store.close()
    rag_count = export_rag(database, rag_dir / "rag_documents.jsonl.gz")
    write_reports(output, reports, summary, rag_count)
    print(json.dumps({"output_root": str(output), "rag_documents": rag_count}, ensure_ascii=False))


In [ ]:
# 처방 문자열을 운동 단계와 개별 운동명으로 분해하고,
# 동일 조건의 반복 기록은 검색 문서에서 occurrence_count로 집계합니다.

def split_prescription(value):
    text = clean_text(value)
    if not text:
        return []
    stage = "단계 미상"
    exercises = []
    for block in re.split(r"[/\n]+", text):
        block = clean_text(block)
        match = re.match(r"^(사전운동|본운동|마무리운동)\s*:\s*(.*)$", block)
        if match:
            stage, block = match.group(1), match.group(2)
        for exercise in re.split(r"[,;]+", block):
            exercise = clean_text(exercise).strip("-· ")
            if len(exercise) >= 2:
                exercises.append((stage, exercise))
    return list(dict.fromkeys(exercises))


_BASE_RAG_ADD = RagStore.add
_BASE_RAG_CLOSE = RagStore.close


def aggregated_rag_add(self, dataset, title, content, metadata, key_parts):
    if dataset not in {"general_prescription", "disability_prescription", "measurement_prescription"}:
        return _BASE_RAG_ADD(self, dataset, title, content, metadata, key_parts)
    if not hasattr(self, "_exercise_cache"):
        self._exercise_cache = {}
    prescription = key_parts[0] if key_parts else ""
    for stage, exercise in split_prescription(prescription):
        item_metadata = dict(metadata)
        item_metadata["exercise_name"] = exercise
        item_metadata["exercise_stage"] = stage
        item_title = f"{exercise} ({stage})"
        item_content = "\n".join([
            item_title,
            f"운동 단계: {stage}",
            f"운동명: {exercise}",
            *[
                line for line in content.splitlines()
                if not line.startswith("처방 내용:") and line != title
            ],
        ])
        item_key_parts = [exercise, stage, *key_parts[1:]]
        normalized = [clean_text(part).lower() for part in item_key_parts]
        doc_key = "␟".join([dataset, *normalized])
        cached = self._exercise_cache.get(doc_key)
        if cached:
            cached[4] += 1
        else:
            self._exercise_cache[doc_key] = [
                dataset, item_title, item_content, item_metadata, 1, item_key_parts
            ]


def aggregated_rag_close(self):
    cache = getattr(self, "_exercise_cache", {})
    for index, (doc_key, values) in enumerate(cache.items(), 1):
        dataset, title, content, metadata, count, _ = values
        doc_id = hashlib.sha256(doc_key.encode("utf-8")).hexdigest()[:24]
        self.connection.execute(
            """INSERT INTO documents(
                id,doc_key,dataset,title,content,metadata_json,occurrence_count
            ) VALUES(?,?,?,?,?,?,?)
            ON CONFLICT(doc_key)
            DO UPDATE SET occurrence_count=excluded.occurrence_count""",
            (
                doc_id, doc_key, dataset, title, content,
                json.dumps(metadata, ensure_ascii=False, sort_keys=True), count,
            ),
        )
        if index % 10000 == 0:
            self.connection.commit()
            print(f"[검색 문서 저장] {index:,}/{len(cache):,}", flush=True)
    self.connection.commit()
    _BASE_RAG_CLOSE(self)


def run_preprocessing(source_root: Path, output_root: Path):
    RagStore.add = aggregated_rag_add
    RagStore.close = aggregated_rag_close
    run(source_root, output_root)


### 2-2. 원본 재전처리(선택)

기존 `rag_documents.sqlite`를 사용하면 이 단계는 건너뜁니다. 원본 6개 폴더로 DB를 다시 만들 때만 `RUN_PREPROCESSING=True`로 실행하세요.

In [ ]:
quality_report_path = ARTIFACTS_DIR / "quality_report.json"

if RUN_PREPROCESSING:
    if missing_folders:
        raise FileNotFoundError(f"누락된 원본 폴더: {missing_folders}")
    run_preprocessing(SOURCE_ROOT, OUTPUT_ROOT)
    ARTIFACTS_DIR = OUTPUT_ROOT / "rag"
    RAG_DATABASE_PATH = ARTIFACTS_DIR / "rag_documents.sqlite"
    quality_report_path = OUTPUT_ROOT / "reports" / "quality_report.json"
    ARTIFACTS_READY = RAG_DATABASE_PATH.is_file()
    print("전처리 완료:", ARTIFACTS_DIR)
else:
    print("전처리를 건너뛰고 기존 DB를 사용합니다:", RAG_DATABASE_PATH)

### 2-3. 전처리 전후 품질검사

행 보존식 `입력 = 정상·경고 + 격리`를 확인하고, 데이터 묶음별 격리·경고 현황을 표와 그래프로 표시합니다.


In [ ]:
if quality_report_path.exists():
    quality = json.loads(quality_report_path.read_text(encoding="utf-8"))
    totals = quality["totals"]
    summary = pd.DataFrame([{
        "입력 행": totals["input_rows"],
        "정상·경고 행": totals["clean_rows"],
        "격리 행": totals["quarantine_rows"],
        "검색 문서": totals["rag_documents"],
        "행 보존 검사": totals["row_conservation_ok"],
    }])
    display(summary)
    display(pd.DataFrame(quality["files"]))
    if not totals["row_conservation_ok"]:
        raise RuntimeError("행 보존 검사 실패: 입력 행이 정상+격리 행과 다릅니다.")
else:
    print("품질 보고서는 선택 파일입니다. RAG DB 실행에는 지장이 없습니다.")

### 2-4. 격리 데이터 표본 확인

삭제된 것처럼 사라지지 않도록 모든 이상 행은 `quarantine` 파일에 원본 필드, 원본 파일명, 원본 행 번호, 격리 사유와 함께 남깁니다.


In [ ]:
def preview_quarantine(processed_dir: Path, limit_per_file: int = 3) -> pd.DataFrame:
    rows = []
    if not processed_dir.exists():
        return pd.DataFrame()
    for path in sorted(processed_dir.glob("*quarantine*")):
        if path.suffix == ".csv" or path.name.endswith(".csv.gz"):
            frame = pd.read_csv(path, nrows=limit_per_file, low_memory=False)
            frame.insert(0, "quarantine_file", path.name)
            rows.extend(frame.to_dict("records"))
        elif path.name.endswith(".jsonl.gz"):
            with gzip.open(path, "rt", encoding="utf-8") as stream:
                for _, line in zip(range(limit_per_file), stream):
                    item = json.loads(line)
                    item["quarantine_file"] = path.name
                    rows.append(item)
    return pd.DataFrame(rows)


if RUN_PREPROCESSING:
    quarantine_preview = preview_quarantine(OUTPUT_ROOT / "processed")
    display(quarantine_preview if not quarantine_preview.empty else "격리 표본 없음")
else:
    print("기존 DB 모드에서는 격리 원본 파일 표본을 건너뜁니다.")

## Step 3. 기존 DB 연결·무결성 검사

`rag_documents.sqlite`를 읽기 전용으로 열어 문서 수와 데이터셋 구성을 확인합니다.  
`analytics.duckdb`가 있으면 `run_analytics()`로 통계 SQL도 실행할 수 있습니다.

In [ ]:
import json
import shutil
import sqlite3
from pathlib import Path

import pandas as pd


# Step 3만 따로 실행해도 로컬 DB를 찾도록 보완합니다.
if "RAG_DATABASE_PATH" not in globals():
    ARTIFACTS_DIR = Path.cwd() / "artifacts"
    RAG_DATABASE_PATH = ARTIFACTS_DIR / "rag_documents.sqlite"

# ------------------------------------------------------------
# 1. Step 1에서 찾은 RAG DB 확인
# ------------------------------------------------------------

if RAG_DATABASE_PATH is None:
    raise FileNotFoundError(
        "RAG DB가 연결되지 않았습니다.\n"
        "Step 1을 다시 실행하고 rag_documents.sqlite를 선택하세요."
    )

RAG_DATABASE_PATH = Path(RAG_DATABASE_PATH)

if not RAG_DATABASE_PATH.is_file():
    raise FileNotFoundError(
        f"RAG DB 파일이 없습니다: {RAG_DATABASE_PATH}"
    )

print("발견한 RAG DB:", RAG_DATABASE_PATH)
print(
    "원본 DB 크기:",
    f"{RAG_DATABASE_PATH.stat().st_size / 1024**2:,.2f} MB",
)


# ------------------------------------------------------------
# 2. Google Drive DB를 Colab 로컬 디스크로 복사
# ------------------------------------------------------------
# Google Drive에 있는 SQLite를 직접 조회하면 매우 느릴 수 있습니다.
# /content/artifacts로 복사한 다음 로컬 파일을 사용합니다.

def copy_with_progress(
    source: Path,
    destination: Path,
    chunk_size: int = 8 * 1024 * 1024,
) -> None:
    """대용량 파일을 복사하면서 진행률을 표시합니다."""

    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(
        destination.suffix + ".copying"
    )

    if temporary.exists():
        temporary.unlink()

    total_size = source.stat().st_size
    copied_size = 0

    with source.open("rb") as source_stream:
        with temporary.open("wb") as destination_stream:
            while True:
                chunk = source_stream.read(chunk_size)

                if not chunk:
                    break

                destination_stream.write(chunk)
                copied_size += len(chunk)

                progress = copied_size / total_size * 100
                print(
                    f"\rDB 복사 중: {progress:6.2f}% "
                    f"({copied_size / 1024**2:,.1f} / "
                    f"{total_size / 1024**2:,.1f} MB)",
                    end="",
                    flush=True,
                )

    temporary.replace(destination)
    print("\nDB 복사 완료:", destination)


source_rag_database = RAG_DATABASE_PATH

if str(source_rag_database).startswith("/content/drive/"):
    local_directory = Path("/content/artifacts")
    local_database = local_directory / "rag_documents.sqlite"

    source_size = source_rag_database.stat().st_size

    local_database_ready = (
        local_database.is_file()
        and local_database.stat().st_size == source_size
    )

    if local_database_ready:
        print("기존 Colab 로컬 DB를 재사용합니다.")
    else:
        copy_with_progress(
            source_rag_database,
            local_database,
        )

    RAG_DATABASE_PATH = local_database
else:
    print("DB가 Colab 로컬 또는 로컬 Jupyter에 있어 복사를 생략합니다.")


# ------------------------------------------------------------
# 3. 실제 사용할 파일 경로 설정
# ------------------------------------------------------------

RAG_DATABASE = RAG_DATABASE_PATH

# analytics.duckdb와 manifest.json은 기존 artifacts 폴더에서 찾습니다.
ANALYTICS_DATABASE = ARTIFACTS_DIR / "analytics.duckdb"
MANIFEST_PATH = ARTIFACTS_DIR / "manifest.json"

print("\n실제 사용할 RAG DB:", RAG_DATABASE)
print(
    "사용 DB 크기:",
    f"{RAG_DATABASE.stat().st_size / 1024**2:,.2f} MB",
)


# ------------------------------------------------------------
# 4. SQLite 읽기 전용 URI
# ------------------------------------------------------------

def sqlite_readonly_uri(path: Path) -> str:
    """
    Windows 한글 경로와 Colab 경로에서 동작하는
    SQLite 읽기 전용 URI를 만듭니다.
    """
    return (
        path.resolve().as_uri()
        + "?mode=ro&immutable=1"
    )


# ------------------------------------------------------------
# 5. SQLite 빠른 무결성 검사 및 문서 수 확인
# ------------------------------------------------------------

with sqlite3.connect(
    sqlite_readonly_uri(RAG_DATABASE),
    uri=True,
) as connection:

    # 전체 integrity_check보다 빠른 검사
    integrity = connection.execute(
        "PRAGMA quick_check(1)"
    ).fetchone()[0]

    document_count = connection.execute(
        "SELECT COUNT(*) FROM documents"
    ).fetchone()[0]

    dataset_counts = pd.read_sql_query(
        """
        SELECT
            dataset,
            COUNT(*) AS documents,
            SUM(occurrence_count) AS occurrences
        FROM documents
        GROUP BY dataset
        ORDER BY dataset
        """,
        connection,
    )


if integrity != "ok":
    raise RuntimeError(
        f"SQLite 무결성 검사 실패: {integrity}"
    )


print("\nSQLite 무결성:", integrity)
print(f"RAG 문서: {document_count:,}건")

display(dataset_counts)


# ------------------------------------------------------------
# 6. manifest 문서 수와 SQLite 문서 수 비교
# ------------------------------------------------------------

if MANIFEST_PATH.is_file():
    manifest = json.loads(
        MANIFEST_PATH.read_text(encoding="utf-8")
    )

    expected_count = (
        manifest
        .get("rag", {})
        .get("document_count")
    )

    if (
        expected_count is not None
        and expected_count != document_count
    ):
        raise RuntimeError(
            f"manifest 문서 수({expected_count:,})와 "
            f"SQLite 문서 수({document_count:,})가 다릅니다."
        )

    print("manifest 문서 수 검사: 정상")
else:
    print(
        "manifest.json 없음: 선택 파일이므로 "
        "RAG 실행에는 문제가 없습니다."
    )


# ------------------------------------------------------------
# 7. DuckDB 통계 조회 함수
# ------------------------------------------------------------

def run_analytics(
    sql: str,
    parameters=None,
) -> pd.DataFrame:
    """
    analytics.duckdb에 SQL을 실행하고
    결과를 DataFrame으로 반환합니다.
    """

    if not ANALYTICS_DATABASE.is_file():
        raise FileNotFoundError(
            f"통계 DB가 없습니다: {ANALYTICS_DATABASE}"
        )

    import duckdb

    with duckdb.connect(
        str(ANALYTICS_DATABASE),
        read_only=True,
    ) as connection:
        return connection.execute(
            sql,
            parameters or [],
        ).fetchdf()


# analytics.duckdb가 있을 때만 아래 코드 실행
#
# display(
#     run_analytics(
#         "SELECT * FROM dataset_catalog ORDER BY dataset"
#     )
# )

## Step 4. Jupyter 패키지와 임베딩 모델

현재 커널의 패키지를 확인하고 배포 폴더에 포함된 `models/ko-sroberta-multitask` 모델을 사용합니다.


In [ ]:
# 설치 셀은 아래 하나로 통합했습니다.

In [ ]:
import importlib.util
import subprocess

required_packages = {
    "sentence_transformers": "sentence-transformers>=5,<7",
    "chromadb": "chromadb>=1,<2",
    "duckdb": "duckdb>=1.4,<2",
}
missing_packages = [
    package for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]

print("현재 커널:", sys.executable)
if missing_packages:
    print("설치:", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
else:
    print("필수 패키지가 이미 설치되어 있습니다.")

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "jhgan/ko-sroberta-multitask"
embedding_model = SentenceTransformer(str(PROJECT_ROOT / "models/ko-sroberta-multitask"))


def embed_texts(texts: list[str]) -> list[list[float]]:
    """코사인 유사도용 정규화 임베딩을 생성합니다."""
    vectors = embedding_model.encode(
        texts,
        batch_size=64,
        show_progress_bar=len(texts) > 64,
        normalize_embeddings=True,
    )
    return vectors.tolist()


print("임베딩 모델 로딩 완료:", EMBEDDING_MODEL_NAME)

## Step 5. 전체 문서 기반 Chroma 갱신

실행 중인 인덱스와 분리된 `chroma_candidate`에 작성합니다. 문서 수뿐 아니라 ID·텍스트·메타데이터를 비교합니다. 기존 문서와 같으면 재사용하고 변경분만 다시 임베딩합니다. 완료 표식에는 원문 DB 해시가 기록됩니다. 검증 후 서버를 정지하고 배포 인덱스와 실행 캐시에 함께 반영해야 합니다.


In [ ]:
import chromadb
import sys
sys.path.insert(0, str(PROJECT_ROOT / "tools"))
from build_full_chroma import build
CHROMA_DIR = RUNTIME_ROOT / "chroma_candidate"
COLLECTION_NAME = "fitness_rag_ko_sroberta_v1"
LIMIT_DOCS = None
BATCH_SIZE = 128
AUTO_BUILD_CHROMA = False  # 갱신할 때만 True. 현재 실행 캐시를 자동 덮어쓰지 않습니다.
if AUTO_BUILD_CHROMA:
    build(RAG_DATABASE, CHROMA_DIR, BATCH_SIZE)
else:
    CHROMA_DIR = RUNTIME_ROOT / "chroma"
if not (CHROMA_DIR / "BUILD_COMPLETE.json").is_file():
    raise RuntimeError("완료된 인덱스가 없습니다. 후보 폴더 구축을 먼저 완료하세요.")
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_collection(COLLECTION_NAME)
with sqlite3.connect(sqlite_readonly_uri(RAG_DATABASE), uri=True) as con:
    expected_ids = {r[0] for r in con.execute("SELECT id FROM documents")}
actual_ids = set(collection.get(include=[])["ids"])
if actual_ids != expected_ids:
    raise RuntimeError(f"ID 불일치: 누락 {len(expected_ids-actual_ids)}, 추가 {len(actual_ids-expected_ids)}")
print("전체 문서 ID 일치:", len(actual_ids), "경로:", CHROMA_DIR)

from rag_index_contract import validate_index
validate_index(RAG_DATABASE, CHROMA_DIR, collection, expected_ids)


## Step 6. 검색 함수 (유사도 임계값 포함)

코사인 유사도가 `MIN_SIMILARITY_SCORE` 미만이면 근거로 인정하지 않습니다. Chroma는 `distance`(작을수록 가까움)를 반환하므로 `score = 1 - distance`로 환산합니다.

`rag_documents.sqlite`에서 상세 필드(운동 조건, 출처 파일 등)를 다시 찾아와 최종 답변에 쓸 메타데이터를 완성합니다.


In [ ]:
DATASET_LABELS = {
    "disability_prescription": "장애인 체력 측정별 운동처방",
    "general_prescription": "체력 측정별 운동처방",
    "measurement_prescription": "국민체력100 측정 기반 운동처방",
    "video_content": "국민체력100 동영상",
}

# 팀 질문 세트로 점수 분포를 확인한 뒤 재조정하세요.
MIN_SIMILARITY_SCORE = 0.35


def _fetch_full_records(ids: list[str]) -> dict:
    if not ids:
        return {}
    with sqlite3.connect(sqlite_readonly_uri(RAG_DATABASE), uri=True) as connection:
        connection.row_factory = sqlite3.Row
        placeholders = ",".join("?" for _ in ids)
        rows = connection.execute(
            f"SELECT id,dataset,title,content,metadata_json,occurrence_count "
            f"FROM documents WHERE id IN ({placeholders})",
            ids,
        ).fetchall()
    return {row["id"]: row for row in rows}


def semantic_search(
    question: str,
    k: int = 5,
    datasets: list[str] | None = None,
    min_score: float = MIN_SIMILARITY_SCORE,
) -> list[dict]:
    """질문을 임베딩해 Chroma에서 검색하고 SQLite에서 상세 필드를 복원합니다."""
    indexed_count = collection.count()
    if indexed_count == 0:
        raise RuntimeError("Chroma가 비어 있습니다. build_chroma_index()를 먼저 실행하세요.")

    query_vector = embed_texts([question])[0]
    where = {"dataset": {"$in": datasets}} if datasets else None
    raw = collection.query(
        query_embeddings=[query_vector],
        n_results=min(max(k * 5, 20), indexed_count),
        where=where,
    )
    ids = raw["ids"][0]
    distances = raw["distances"][0]
    full_records = _fetch_full_records(ids)

    results = []
    for doc_id, distance in zip(ids, distances):
        score = 1.0 - float(distance)
        if score < min_score:
            continue
        record = full_records.get(doc_id)
        if record is None:
            continue
        results.append({
            "id": doc_id,
            "dataset": record["dataset"],
            "title": record["title"],
            "text": record["content"],
            "metadata": json.loads(record["metadata_json"]),
            "occurrence_count": record["occurrence_count"],
            "score": score,
        })
    results.sort(key=lambda item: item["score"], reverse=True)
    return results[:k]


# 검색 테스트
search_test = semantic_search("악력을 높이는 운동을 알려줘", k=3)
display(pd.DataFrame([
    {"score": round(item["score"], 4), "dataset": item["dataset"], "title": item["title"]}
    for item in search_test
]))

## Step 7. 코치 페르소나 프롬프트 + LLM 생성

`build_prompt()`가 검색된 근거만으로 프롬프트를 구성합니다. "컨텍스트에 없으면 답하지 말 것"을 명시해 근거 밖 생성(환각)을 최대한 억제합니다.

`generate_answer()`는 `llm_call`을 인자로 받는 구조라, 실제 LLM을 연결하기 전에도(API 키가 없어도) 파이프라인 전체가 정상 동작하는지 먼저 확인할 수 있습니다.


In [ ]:
import urllib.request
def generate_answer(question, k=5, datasets=None, llm_call=None, profile=None):
    """배포 웹 서버의 Qwen3 하네스 경로로 질문합니다."""
    if llm_call is not None or datasets is not None:
        raise ValueError("별도 LLM/데이터 범위는 지원하지 않습니다. 배포 하네스를 사용합니다.")
    payload = {**(profile or {}), "user_id":"notebook-test", "question":question, "selection_mode":False}
    request = urllib.request.Request("http://127.0.0.1:8501/api/coach",
        data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
        headers={"Content-Type":"application/json"}, method="POST")
    with urllib.request.urlopen(request, timeout=240) as response:
        return json.load(response)
# result = generate_answer("악력을 높이는 운동을 알려줘")
# 조건이 필요한 추천은 profile={"age":30, "sex":"M", "target_area":"허리", "exercise_type":"스트레칭", "fitness_level":"종합", "equipment":"없음"} 등을 명시하세요.
# 조건/근거가 부족하면 하네스가 추천을 보류할 수 있습니다.
# display(result)


## Step 7-1. Qwen3 하네스

`START_AI_FITNESS_SERVER.cmd`로 배포 웹 서버를 먼저 실행하세요. Step 7의 `generate_answer`는 `/api/coach`를 호출하므로 현재 배포본의 후보 검증·출처·영상·안전 규칙을 적용합니다. 모의 응답을 실제 LLM 응답으로 표시하지 않습니다.


## Step 8. 검증

Step 3에서 SQLite 무결성·문서 수, Step 5에서 전체 문서 ID 일치를 확인합니다. Step 6은 검색 확인, Step 7은 실제 Qwen3 하네스 응답 확인입니다. 재구축 시 `AUTO_BUILD_CHROMA=True`로 후보 폴더에 전체 문서를 갱신하고 검증 후 반영합니다.
